# Etapa 5 — pre-experiment diagnostic

Diagnóstico de prontidão para reference list reconstruction, não experimento de desempenho.
Não há embeddings reais persistidos nem OPENAI_API_KEY na inspeção de 2026-09-28.
Este notebook não chama o provider e não fabrica métricas híbridas.

Ground truth: referências diretas ∩ IDs com metadata, excluindo a própria query.
Candidatos: REFERENCE LIST completa (profundidades 1–3), sem unir CITATION LIST.
Candidatos sem abstract continuam elegíveis; IDs sem metadata são excluídos.
Queries precisam de metadata, grafo, abstract, ground truth não vazio e >=20 candidatos
elegíveis, com ao menos um abstract de candidato para permitir a imputação existente.
Não há classificação review/non-review no dataset disponibilizado.

Métricas validadas em testes: hit@k = relevantes no top-k / k; recall@20 = relevantes
no top-20 / total relevante. Exigem IDs únicos, ranking suficientemente longo e
ground truth não vazio. A ambiguidade do artigo está em docs/decisions.md.
Pesos previstos: abstract=0.6, title=0.2, citation=0.2. Nenhum score será calculado aqui.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from literature_recommender.data import load_papers, load_citations
from literature_recommender.graph import build_citation_graph
from literature_recommender.experiment import (
    reference_query_population, select_query_sample, prepare_reference_query,
)

assert sys.version_info[:2] == (3, 11), "Use o ambiente Python 3.11 do projeto"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data/raw/Academic-Literature-Recommendation-in-Large-scale-Citation-Networks-main"
SEED = 42
MAX_QUERIES = 5
WEIGHTS = (0.6, 0.2, 0.2)
print({"seed": SEED, "max_queries": MAX_QUERIES, "weights": WEIGHTS,
       "api_key_present": bool(os.environ.get("OPENAI_API_KEY")),
       "provider_abstracts_sent": 0, "n_evaluated": 0})

In [ ]:
papers = load_papers(RAW / "article_info.xlsx")
citations = load_citations(RAW / "citation_network.csv")
graph = build_citation_graph(citations)
raw_columns = pd.read_excel(RAW / "article_info.xlsx", nrows=0).columns.tolist()
print({"raw_columns": raw_columns, "papers": len(papers),
       "citation_rows": len(citations), "graph_nodes": len(graph),
       "missing_abstracts": int(papers.abstract.isna().sum()),
       "duplicate_paper_ids": int(papers.paper_id.duplicated().sum()),
       "invalid_titles": int((~papers.title.str.contains(r"\w", na=False)).sum()),
       "review_classification_available": False})
print(papers.dtypes.to_string())

In [ ]:
# Enumera a população inteira antes de impor qualquer limite de custo.
population = reference_query_population(papers, graph)
print({"eligible_population_before_cost_cap": len(population)})
print(population.n_eligible_candidates.describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).to_string())
print({cap: int(population.n_eligible_candidates.le(cap).sum())
       for cap in [100, 200, 500, 1000]})

In [ ]:
# Limite definido depois de inspecionar a distribuição completa:
# 45.397 / 107.054 queries (~42,4%) têm 20–100 candidatos elegíveis.
# Favorece pools menores; não representa toda a população.
MAX_CANDIDATES = 100
sampling_population = population[population.n_eligible_candidates.le(MAX_CANDIDATES)]
selected = select_query_sample(
    population, seed=SEED, max_queries=MAX_QUERIES, max_candidates=MAX_CANDIDATES,
)
plans = [prepare_reference_query(papers, graph, query_id) for query_id in selected]
diagnostic = pd.DataFrame(
    [plan["diagnostic"] for plan in plans],
    columns=["query_id", "n_ground_truth", "n_structural_candidates",
             "n_eligible_candidates", "n_candidates_without_abstract",
             "n_candidates_without_metadata", "n_unique_abstracts_needed"],
)
# Textos repetidos entre queries seriam enviados apenas uma vez num futuro run.
unique_texts = sorted({text for plan in plans for text in plan["abstracts"].values()
                       if text is not None})
print({"mode": "pre-experiment diagnostic", "seed": SEED,
       "eligible_population_before_cost_cap": len(population),
       "sampling_population": len(sampling_population), "n_queries": len(selected),
       "n_structural_candidates_total": int(diagnostic.n_structural_candidates.sum()),
       "n_eligible_candidates_total": int(diagnostic.n_eligible_candidates.sum()),
       "n_unique_abstracts_needed": len(unique_texts),
       "n_evaluated": 0, "provider_abstracts_sent": 0})
print(diagnostic.to_string(index=False))
print("Exclusões:", {plan["query_id"]: plan["excluded"] for plan in plans})

In [ ]:
# Somente contagens derivadas e configuração; não salva textos ou embeddings.
output = ROOT / "data/processed/preliminary_diagnostic.csv"
output.parent.mkdir(parents=True, exist_ok=True)
diagnostic.assign(
    mode="pre-experiment diagnostic", seed=SEED, max_candidates=MAX_CANDIDATES,
    n_population=len(population), n_sampling_population=len(sampling_population),
).to_csv(output, index=False)
print(output.relative_to(ROOT))

## Registro do diagnóstico — 2026-09-28

Resultados obtidos por execução direta das mesmas APIs no Python 3.11.16 configurado.
A IDE não iniciou o kernel: a célula de setup permaneceu na fila com estado down.
As células foram inspecionadas, mas não há execução completa do notebook validada;
esta tabela é um registro explícito da execução externa, não saída fabricada de célula.

População elegível: **107.054**. Após limite de 20–100 candidatos: **45.397**.
Seed **42**, amostra **5**. Mediana de candidatos na população completa: **127**;
p75 **283**, máximo **4.297**. O limite preserva **42,4%** da população e favorece
pools menores. Não houve truncamento de qualquer lista selecionada.

| query_id | ground truth | estruturais | elegíveis | sem abstract | sem metadata | abstracts únicos |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| paper_84898 | 5 | 52 | 52 | 45 | 0 | 8 |
| paper_132059 | 10 | 76 | 74 | 43 | 2 | 32 |
| paper_105189 | 6 | 100 | 100 | 16 | 0 | 85 |
| paper_167605 | 9 | 90 | 77 | 4 | 13 | 74 |
| paper_15976 | 3 | 86 | 86 | 84 | 0 | 3 |

Totais: **404** candidatos estruturais, **389** elegíveis (355 IDs distintos),
**199 abstracts únicos** incluindo queries. Todas as 15 exclusões são
missing_metadata. Abstracts ausentes permanecem elegíveis, inclusive o caso
84/86, mantido no sorteio sem substituição.

**N avaliado = 0; abstracts enviados ao provider = 0.** Nenhum embedding,
ranking híbrido, hit/recall real ou média de desempenho foi calculado.
Arquivo: data/processed/preliminary_diagnostic.csv. Não há classificação
review/non-review; este diagnóstico não reproduz os resultados dos autores.